# TP4 — Knowledge Base Construction, Alignment & Expansion
**Domain : Science-Fiction (authors & works)**

Ce notebook importe les fonctions depuis `src/kg/kg_builder.py`.

In [4]:
import warnings
warnings.filterwarnings("ignore")

import os
os.environ["PYTHONWARNINGS"] = "ignore"
import numpy as np
np.warnings = None

import sys
sys.path.append("../")

import time
import pandas as pd
from rdflib.namespace import RDF, RDFS, OWL, XSD
from rdflib import Literal, URIRef

from src.kg.kg_builder import (
    init_graph, PRIV, PRED, DBO,
    fetch_books_from_api, build_initial_kg,
    link_entity_sparql, link_entity_to_dbpedia,
    search_dbo_properties,
    get_scifi_anchors, expand_kg, clean_kg, print_kg_stats
)

## Step 1 — KB initiale via Open Library API

In [5]:
my_private_kg = init_graph()

books = fetch_books_from_api(subject="science_fiction", limit=35)
my_private_kg = build_initial_kg(books, my_private_kg)

print(f"Triplets  : {len(my_private_kg)}")

Fetching: https://openlibrary.org/subjects/science_fiction.json?limit=35
Triplets  : 190


## Step 2 — Entity Linking avec DBpedia

In [6]:
wiki_urls = [
    "https://en.wikipedia.org/wiki/Isaac_Asimov",
    "https://en.wikipedia.org/wiki/Frank_Herbert",
    "https://en.wikipedia.org/wiki/Arthur_C._Clarke",
    "https://en.wikipedia.org/wiki/Philip_K._Dick",
    "https://en.wikipedia.org/wiki/Ursula_K._Le_Guin",
    "https://en.wikipedia.org/wiki/Science_fiction",
    "https://en.wikipedia.org/wiki/Dune_(novel)",
]

mapping_table = []
for wiki_url in wiki_urls:
    page_name   = wiki_url.split("/wiki/")[-1]
    private_uri = PRIV[page_name]
    dbpedia_uri, conf = link_entity_sparql(wiki_url)
    if dbpedia_uri:
        my_private_kg.add((private_uri, OWL.sameAs, URIRef(dbpedia_uri)))
    mapping_table.append({"Private Entity": page_name, "External URI": dbpedia_uri or "Not Found", "Confidence": conf})
    time.sleep(0.2)

print(f"{'Private Entity':<30} | {'External URI':<55} | Confidence")
print("-" * 95)
for row in mapping_table:
    print(f"{row['Private Entity']:<30} | {row['External URI']:<55} | {row['Confidence']}")

Erreur pour http://dbpedia.org/resource/Arthur_C._Clarke : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)
Erreur pour http://dbpedia.org/resource/Science_fiction : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)
Private Entity                 | External URI                                            | Confidence
-----------------------------------------------------------------------------------------------
Isaac_Asimov                   | http://dbpedia.org/resource/Isaac_Asimov                | 1.0
Frank_Herbert                  | http://dbpedia.org/resource/Frank_Herbert               | 1.0
Arthur_C._Clarke               | Not Found                                               | 0.0
Philip_K._Dick                 | http://dbpedia.org/resource/Philip_K._Dick              | 1.0
Ursula_K._Le_Guin              | http://dbpedia.org/resource/Ursula_K._Le_Guin           | 1.0
Science_fiction                | Not Fou

## Step 2 (suite) — Entité non trouvée → Définition ontologique

In [7]:
my_private_kg.add((PRIV.SciFiFanzine, RDFS.subClassOf, DBO.PeriodicalLiterature))
my_private_kg.add((PRIV.GalaxyMagazine1950, RDF.type,       PRIV.SciFiFanzine))
my_private_kg.add((PRIV.GalaxyMagazine1950, PRED.title,     Literal("Galaxy Science Fiction", datatype=XSD.string)))
my_private_kg.add((PRIV.GalaxyMagazine1950, PRED.foundedIn, Literal(1950, datatype=XSD.integer)))
my_private_kg.add((PRED.foundedIn, RDFS.domain, PRIV.SciFiFanzine))
my_private_kg.add((PRED.foundedIn, RDFS.range,  XSD.integer))
print(f"Total triplets : {len(my_private_kg)}")

Total triplets : 201


## Step 3 — Alignement de prédicats

In [8]:
for keyword in ["author", "year", "title", "name", "work", "genre"]:
    results = search_dbo_properties(keyword)
    print(f"\nKeyword '{keyword}' → {len(results)} candidat(s) :")
    for c in results[:3]:
        print(f"  {c['URI']}  |  {c['Label']}")

Erreur : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)

Keyword 'author' → 0 candidat(s) :
Erreur : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)

Keyword 'year' → 0 candidat(s) :
Erreur : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)

Keyword 'title' → 0 candidat(s) :

Keyword 'name' → 10 candidat(s) :
  http://dbpedia.org/ontology/fuelTypeName  |  fuel type
  http://dbpedia.org/ontology/alternativeName  |  alternative name
  http://dbpedia.org/ontology/birthName  |  birth name
Erreur : HTTPSConnectionPool(host='dbpedia.org', port=443): Read timed out. (read timeout=10)

Keyword 'work' → 0 candidat(s) :

Keyword 'genre' → 5 candidat(s) :
  http://dbpedia.org/ontology/genre  |  genre
  http://dbpedia.org/ontology/literaryGenre  |  literary genre
  http://dbpedia.org/ontology/musicFusionGenre  |  music fusion genre


In [9]:
my_private_kg.add((PRED.title,           OWL.equivalentProperty, DBO.title))
my_private_kg.add((PRED.genre,           OWL.equivalentProperty, DBO.genre))
my_private_kg.add((PRED.firstPublishYear,RDFS.subPropertyOf,     DBO.activeYears))
my_private_kg.add((PRED.name,            RDFS.subPropertyOf,     DBO.alternativeName))
my_private_kg.add((PRED.wroteBook,       OWL.inverseOf,          DBO.author))
print(f"Total triplets : {len(my_private_kg)}")

Total triplets : 206


## Step 4 — Expansion 2-hop depuis 300 auteurs Sci-Fi DBpedia

In [10]:
anchors  = get_scifi_anchors(limit=300)
print(f"{len(anchors)} ancres trouvées. Expansion en cours...")

final_kg = expand_kg(anchors, sleep=0.1)
print_kg_stats(final_kg, "KG après expansion")

300 ancres trouvées. Expansion en cours...
  10/300 — triplets : 912
  20/300 — triplets : 6,054
  30/300 — triplets : 8,304
  40/300 — triplets : 9,703
  50/300 — triplets : 12,119
  60/300 — triplets : 13,687
  70/300 — triplets : 14,716
  80/300 — triplets : 15,900
  90/300 — triplets : 16,934
  100/300 — triplets : 18,628
  110/300 — triplets : 19,656
  120/300 — triplets : 22,227
  130/300 — triplets : 24,765
  140/300 — triplets : 26,116
  150/300 — triplets : 27,307
  160/300 — triplets : 30,228
  170/300 — triplets : 34,577
  180/300 — triplets : 39,114
  190/300 — triplets : 41,091
  200/300 — triplets : 43,123
  210/300 — triplets : 44,256
  220/300 — triplets : 45,927
  230/300 — triplets : 46,775
  240/300 — triplets : 51,377
  250/300 — triplets : 58,212
  260/300 — triplets : 63,671
  270/300 — triplets : 67,315
  280/300 — triplets : 70,619
  290/300 — triplets : 72,859
  300/300 — triplets : 74,989

KG après expansion
Triplets   :   75,163  (cible : 50k–200k)
Entités   

## Step 5 — Nettoyage & Export

In [11]:
clean_kg_graph = clean_kg(final_kg, top_n_predicates=150)
print_kg_stats(clean_kg_graph, "KG après nettoyage")

clean_kg_graph.serialize(destination="final_expanded_kg.nt", format="nt", encoding="utf-8")
print("Exporté → final_expanded_kg.nt")


KG après nettoyage
Triplets   :   48,317  (cible : 50k–200k)
Entités    :   26,243  (cible : 5k–30k)
Prédicats  :       72  (cible : 50–200)
Exporté → final_expanded_kg.nt


## Step 6 — Entity Linking depuis extracted_knowledge_scifi.csv

In [12]:
df = pd.read_csv("extracted_knowledge_scifi.csv")
relevant_entities = df[df['type'].isin(['PERSON', 'WORK_OF_ART'])]['entity'].dropna().unique()
print(f"Entités à traiter : {len(relevant_entities)}")

mapping_results = []
for idx, entity in enumerate(relevant_entities):
    if idx > 0 and idx % 20 == 0:
        print(f"  {idx}/{len(relevant_entities)}")
    uri, conf = link_entity_to_dbpedia(entity)
    mapping_results.append({"Private Entity": entity, "External URI": uri or "Not Found", "Confidence": conf})
    if uri:
        private_uri = PRIV[entity.replace(" ", "_")]
        clean_kg_graph.add((private_uri, OWL.sameAs, URIRef(uri)))
    time.sleep(0.1)

df_mapping = pd.DataFrame(mapping_results)
df_mapping.to_csv("entity_alignment.csv", index=False)
found = df_mapping[df_mapping["Confidence"] > 0]
print(f"\nEntités alignées : {len(found)}/{len(relevant_entities)}")

clean_kg_graph.serialize(destination="final_expanded_kg.nt", format="nt", encoding="utf-8")
print(f"Graphe final : {len(clean_kg_graph):,} triplets → final_expanded_kg.nt")

Entités à traiter : 230
  20/230
  40/230
  60/230
  80/230
  100/230
  120/230
  140/230
  160/230
  180/230
  200/230
  220/230

Entités alignées : 164/230
Graphe final : 48,481 triplets → final_expanded_kg.nt
